# In-class Exercise: Anomaly Detection with Network Data



- Explore a real network intrusion dataset (CIC-IDS2017).
- Build and evaluate simple unsupervised anomaly detectors (e.g., OneClassSVM, IsolationForest).
- Compare unsupervised detection performance across different days and attack mixes.
- Interpret detection results: true/false positives and practical trade-offs.



### Overview of the dataset

Dataset Characteristics: CIC-IDS2017 dataset contains network traffic data for the development and evaluation of intrusion detection systems. The dataset is designed to be representative of modern network traffic and includes more than 2.8 million network packets captured over a period of seven days in a real network environment. The dataset includes normal traffic and seven different attack scenarios: Brute Force, Heartbleed, Botnet, DoS, DDoS, Web Attack and Infiltration. The dataset consists of 2830743 rows and 79 columns. In these columns, 78 of them are features that are numerical and the 'label' column is categorical

## Attack types in this dataset (Knowing the attack type is not critical to attempt this in-class exercise. The information is for your knowledge)

1) Brute Force (FTP-Patator, SSH-Patator): Repeated authentication attempts against services (FTP, SSH) to guess credentials.
- Network indicators: Many small flows to the same server/port, repeated failed connection attempts, short durations, similar packet sizes, increased session counts from same source IP.

2) Heartbleed: Exploit of an OpenSSL vulnerability causing data leakage and abnormal packet payloads.
- Network indicators: Unexpected large responses, odd TLS payload patterns or timing anomalies, flows with unusual byte patterns.

3) Bot (Botnet traffic): Compromised hosts communicating with C2 (command-and-control) servers, periodic beacons, or participating in coordinated attacks.
- Network indicators: Regular, periodic small transfers to one or several external hosts, DNS anomalies, uncommon ports, or synchronized behavior across hosts.


4) DoS (Denial of Service) and DDoS: Attacker floods target(s) with traffic to disrupt service; DDoS involves many distributed sources.
- Network indicators: Very large flows (bytes, packets), extremely high packet rates, many short flows from many sources to same destination (for DDoS).

5) Web Attacks (SQLi, XSS, Web brute force): Attacks targeting web applications via malformed HTTP requests, brute forcing web logins, or injection payloads.
- Network indicators: Large numbers of HTTP requests to specific URLs or with unusual payloads, repeated POST/GETs, spikes in request rates to web servers.

6) Infiltration: Attackers who successfully penetrate the network (post-exploitation activities, lateral movement).
- Network indicators: Unusual SMB/RDP/other service connections, new destinations contacted by internal hosts, longer duration flows with larger bytes transferred.


7) Port Scan: Scanning many ports on a host to discover open services.
- Network indicators: Many short flows from one source to many destination ports, low bytes per flow, sequential or random port access patterns.


Data [link](https://csciitd-my.sharepoint.com/:f:/g/personal/tmangla_iitd_ac_in/EoNOWmPMifxDsInZccN-pugB_MQYztP4UWuyHrlV_N5LJQ?e=52ticB)

# Step 1: Load the data 

In [ ]:
import pandas as pd
import os
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
import seaborn as sns
from glob import glob
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, auc

In [ ]:
def read_data(fname):
    df = pd.read_csv(fname)
    df.rename(columns={col: col.strip() for col in df.columns}, inplace=True)
    return df 
    
def pre_process(df):
    df = df[df.columns.tolist()[:-1]]
    df = df.fillna(0)
    df.replace([np.inf, -np.inf], -1, inplace=True)
    return df

In [ ]:
# Loading the dataset
data1 = read_data('data/CIC-IDS-2017/Monday-WorkingHours.pcap_ISCX.csv.gz')
data2 = read_data('data/CIC-IDS-2017/Tuesday-WorkingHours.pcap_ISCX.csv.gz')
data3 = read_data('data/CIC-IDS-2017//Wednesday-workingHours.pcap_ISCX.csv.gz')
data4 = read_data('data/CIC-IDS-2017/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.gz')
data5 = read_data('data/CIC-IDS-2017/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.gz')
data6 = read_data('data/CIC-IDS-2017/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.gz')

Each row in the dataset corresponds to a network flow, where a flow is defined as a sequence of packets sharing the same 5-tuple (source IP, destination IP, source port, destination port, and protocol). The columns contain network features extracted from these flows (e.g., packet lengths, inter-arrival times, flow duration) as well as a label indicating whether the flow is benign or attack traffic.

In [ ]:

data1.columns

In [ ]:
pd.options.display.max_columns = 80
data1

#### Labels

In [ ]:
data1["Label"].value_counts()

Monday data has no attack traffic. You are encouraged to check out the traffic distribution for other days

# Step 2: Data Characterization

**Exercise**: Read data from Thursday and plot the distribution for the features below for both benign and attack traffic. 

In [ ]:
features = ['Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s']

In [ ]:
fname = "data/CIC-IDS-2017/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.gz"
df = pd.read_data(fname)

In [ ]:
## CODE HERE


# Step 3: Unsupervised learning

**Exercise**: Train an anomaly detection model. Report the TPR, FPR and AUC for each day

In [ ]:
## Code here
data_train = data1 ## Monday data. Note that it has no attack traffic. So it is the most ideal for training. 
data_test = data2 ## Tuesday data 

In [ ]:
## remove nan, inf, -inf val
def preprocess(df):
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df = df.dropna()
    return df

In [ ]:
data_train = preprocess(data_train)
data_test = preprocess(data_test)

In [ ]:
# Creating a dictionary that maps each label to its attack type
attack_map = {
    'BENIGN': 'BENIGN',
    'DDoS': 'DDoS',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'PortScan': 'Port Scan',
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
    'Bot': 'Bot',
    'Web Attack � Brute Force': 'Web Attack',
    'Web Attack � XSS': 'Web Attack',
    'Web Attack � Sql Injection': 'Web Attack',
    'Infiltration': 'Infiltration',
    'Heartbleed': 'Heartbleed'
}
label_col = "Attack Type"
# Creating a new column 'Label1' in the DataFrame based on the attack_map dictionary
data_train['Attack Type'] = data_train['Label'].map(attack_map)
data_train['Type'] = data_train['Attack Type'].apply(lambda x: x if x == "BENIGN" else "ATTACK")

data_test['Attack Type'] = data_test['Label'].map(attack_map)
data_test['Type'] = data_test['Attack Type'].apply(lambda x: x if x == "BENIGN" else "ATTACK")

In [ ]:
print(data_train['Type'].value_counts())
print(data_test['Type'].value_counts())

In [ ]:
# Define features_col: all numeric network feature columns excluding label columns
_exclude = ['Label', 'Attack Type', 'Type']
features_col = [c for c in data_train.columns if c not in _exclude]
# keep only numeric columns (network features)
features_col = data_train[features_col].select_dtypes(include=[np.number]).columns.tolist()

# optional: show count of features
print(f"{len(features_col)} features selected.")

In [ ]:
label_col = 'Type'
# Separate features and labels
X_train = data_train[features_col]  # Training Features
y_train = data_train[label_col]                 # Training Labels
X_test = data_test[features_col]   # Testing Features
y_test = data_test[label_col]   

In [ ]:
## Unsupervised learning models
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.calibration import LabelEncoder



In [ ]:
# Encode the labels (Benign = 0, Attack = 1)
encoder = LabelEncoder()
# ensure encoder maps BENIGN->0, ATTACK->1
encoder.fit(['BENIGN', 'ATTACK'])
y_test_encoded = encoder.transform(y_test)  # Only needed for evaluation
y_train_encoded = encoder.transform(y_train)

# Clean and normalize the data
# Replace inf/-inf with NaN and fill NaNs using training medians (prevents leakage)
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

X_train_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test_clean.replace([np.inf, -np.inf], np.nan, inplace=True)

train_medians = X_train_clean.median()
X_train_clean.fillna(train_medians, inplace=True)
X_test_clean.fillna(train_medians, inplace=True)


# Normalize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)


In [ ]:
# Train an unsupervised anomaly detector  on benign training data


# Get anomaly scores for the test set (higher = more anomalous)
# decision_function gives higher values for inliers, so invert it to get anomaly score


# Predict outliers (-1 indicates outlier)


# Compute metrics: Precision-Recall AUC, TPR, FPRpr_auc = auc(recall, precision)

tp = np.sum((preds == 1) & (y_test_encoded == 1))
fp = np.sum((preds == 1) & (y_test_encoded == 0))
tn = np.sum((preds == 0) & (y_test_encoded == 0))
fn = np.sum((preds == 0) & (y_test_encoded == 1))

tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

print(f"TPR (Recall): {tpr:.4f}")
print(f"FPR: {fpr:.4f}")

# You can adjust IsolationForest 'contamination' or try other detectors (OneClassSVM, LOF)




**Take home**: Can you think of ways to improve the accuracy of detection?